In [ ]:
import wandb
import transformers
import torch
import pandas as pd
import os, glob, re, random
from skimage.metrics import peak_signal_noise_ratio as psnr
from sklearn.decomposition import PCA
from transformers import pipeline
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.cm as cm
import os, glob
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import math
from pathlib import Path

In [ ]:
import os
import sys
import cv2
import torch
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

sapiens_dir = "../Sapiens-Pytorch-Inference"

if not os.path.isdir(sapiens_dir):
    raise FileNotFoundError(f"Sapiens-Ordner nicht gefunden: {sapiens_dir}")

sys.path.insert(0, sapiens_dir)

from sapiens_inference import (
    SapiensDepth,
    SapiensDepthType,
    SapiensSegmentation,
    SapiensSegmentationType,
    SapiensConfig,
)

config = SapiensConfig()
config.depth_type = SapiensDepthType.DEPTH_1B
config.segmentation_type = SapiensSegmentationType.SEGMENTATION_1B

if torch.cuda.is_available():
    config.device = "cuda"
else:
    print("cpu")
    config.device = "cpu"

orig_cwd = os.getcwd()
try:
    os.makedirs(os.path.join(sapiens_dir, "models"), exist_ok=True)
    os.chdir(sapiens_dir)
    depth_predictor = SapiensDepth(config.depth_type, config.device, config.dtype)
    seg_predictor = SapiensSegmentation(config.segmentation_type, config.device, config.dtype)
finally:
    os.chdir(orig_cwd)

print("device", config.device)


def run_sapiens_on_image(img_path):
    pil = Image.open(img_path).convert("RGB")
    rgb_np = np.array(pil)
    bgr_np = cv2.cvtColor(rgb_np, cv2.COLOR_RGB2BGR)
    H, W = bgr_np.shape[:2]

    # Segmentation
    seg_logits = seg_predictor(bgr_np)
    if isinstance(seg_logits, torch.Tensor):
        seg_map = seg_logits.squeeze().cpu().numpy()
    else:
        seg_map = seg_logits

    if seg_map.shape != (H, W):
        seg_map = cv2.resize(seg_map, (W, H), interpolation=cv2.INTER_LINEAR)

    human_mask = (seg_map > 0.5).astype(np.uint8)

    return rgb_np, human_mask

In [ ]:
def run_sapiens_depth_with_bg_far(img_path):
    pil = Image.open(img_path).convert("RGB")
    rgb_np = np.array(pil)
    bgr_np = cv2.cvtColor(rgb_np, cv2.COLOR_RGB2BGR)
    H, W = bgr_np.shape[:2]

    # --- 1) Segmentation für human_mask (sonst kannst du bg nicht setzen)
    seg_logits = seg_predictor(bgr_np)
    if isinstance(seg_logits, torch.Tensor):
        seg_map = seg_logits.squeeze().detach().cpu().numpy()
    else:
        seg_map = np.squeeze(seg_logits)

    if seg_map.shape != (H, W):
        seg_map = cv2.resize(seg_map, (W, H), interpolation=cv2.INTER_LINEAR)

    human_mask = (seg_map > 0.5).astype(np.uint8)

    # --- 2) Depth
    depth_raw = depth_predictor(bgr_np)
    if isinstance(depth_raw, torch.Tensor):
        depth_np = depth_raw.squeeze().detach().cpu().numpy()
    else:
        depth_np = np.squeeze(depth_raw)

    if depth_np.shape != (H, W):
        depth_np = cv2.resize(depth_np, (W, H), interpolation=cv2.INTER_LINEAR)

    # --- 3) Normalisieren (robust gegen NaNs / konstante Maps)
    d_min, d_max = np.nanmin(depth_np), np.nanmax(depth_np)
    depth_norm = (depth_np - d_min) / (d_max - d_min + 1e-8)
    depth_norm = np.clip(depth_norm, 0.0, 1.0)

    # OPTIONAL: falls die Tiefe "falsch herum" ist (nah=hell statt dunkel etc.)
    # depth_norm = 1.0 - depth_norm

    # --- 4) Hintergrund auf "weit weg" setzen
    modified_depth = np.where(human_mask == 1, depth_norm, 1.0)

    # --- 5) Visualisierung
    modified_8u = (modified_depth * 255).astype(np.uint8)
    depth_color_bgr = cv2.applyColorMap(modified_8u, cv2.COLORMAP_TURBO)
    depth_color_rgb = cv2.cvtColor(depth_color_bgr, cv2.COLOR_BGR2RGB)

    return rgb_np, human_mask, depth_norm, depth_color_rgb


In [ ]:
img_path = "pfad/zu/deinem_bild.jpg"

rgb, human_mask, depth_norm, depth_color_rgb = run_sapiens_depth_with_bg_far(img_path)


In [ ]:
plt.figure(figsize=(15,5))

plt.subplot(1,3,1)
plt.title("RGB")
plt.imshow(rgb)
plt.axis("off")

plt.subplot(1,3,2)
plt.title("Human mask")
plt.imshow(human_mask, cmap="gray")
plt.axis("off")

plt.subplot(1,3,3)
plt.title("Depth (colored)")
plt.imshow(depth_color_rgb)
plt.axis("off")

plt.show()